In [71]:
#!#pip install numpy
#!pip install pandas matplotlib seaborn scikit-learn xgboost
#!pip install tensorflow
    
""" TODO : forecast growth calculation 
           XGBRegressor
"""

' TODO : forecast growth calculation \n           XGBRegressor\n'

IMPORTING NECESSARY LIBRARY

In [72]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn import metrics
from sklearn.svm import SVC
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error as mae

import warnings
warnings.filterwarnings('ignore')


dfSET LOADING

In [73]:
df = pd.read_csv('/home/infinity/Documents/IP/retail_store_inventory.csv')
display(df.head())
display(df.tail())


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,20,Snowy,0,72.45,Winter
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,10,Cloudy,0,83.78,Autumn
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,10,Rainy,0,10.91,Winter
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,20,Rainy,0,55.80,Spring
73099,2024-01-01,S005,P0020,Groceries,East,117,6,165,2.33,78.39,20,Rainy,1,79.52,Spring


SHAPE

In [74]:
df.shape

(73100, 15)

DESCRIPTION

In [75]:
df.describe()

,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Holiday/Promotion,Competitor Pricing
count,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000,73100.000000
mean,274.469877,136.464870,110.004473,141.494720,55.135108,10.009508,0.497305,55.146077
std,129.949514,108.919406,52.277448,109.254076,26.021945,7.083746,0.499996,26.191408
min,50.000000,0.000000,20.000000,-9.990000,10.000000,0.000000,0.000000,5.030000
25%,162.000000,49.000000,65.000000,53.670000,32.650000,5.000000,0.000000,32.680000
50%,273.000000,107.000000,110.000000,113.015000,55.050000,10.000000,0.000000,55.010000
75%,387.000000,203.000000,155.000000,208.052500,77.860000,15.000000,1.000000,77.820000
max,500.000000,499.000000,200.000000,518.550000,100.000000,20.000000,1.000000,104.940000


INFO

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  object 
 1   Store ID            73100 non-null  object 
 2   Product ID          73100 non-null  object 
 3   Category            73100 non-null  object 
 4   Region              73100 non-null  object 
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  object 
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  object 
dtypes: float64(3), int64(5), object(7)
memory usage: 8.4+

FEATURE ENGINEERING

DERIVED ATTRIBUTES


In [77]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df.head(1000)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2022-01-10,S005,P0016,Groceries,East,407,103,90,107.56,60.15,15,Sunny,1,64.52,Winter,2022,1,10
996,2022-01-10,S005,P0017,Furniture,East,308,266,164,259.60,31.07,20,Rainy,1,30.57,Summer,2022,1,10
997,2022-01-10,S005,P0018,Furniture,South,331,235,49,234.25,22.58,20,Cloudy,1,27.56,Summer,2022,1,10
998,2022-01-10,S005,P0019,Electronics,East,481,154,169,169.61,85.04,5,Cloudy,0,82.62,Autumn,2022,1,10


In [78]:
#Deleting the Date column
df.drop('Date', axis=1, inplace=True)
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1


CYCLICAL ENCODING

In [79]:
df['M1'] = np.sin(df['Month'] * (2 * np.pi / 12))
df['M2'] = np.cos(df['Month'] * (2 * np.pi / 12))
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1,0.5,0.866025
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1,0.5,0.866025
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1,0.5,0.866025
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1,0.5,0.866025
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1,0.5,0.866025


VALUES ENCODING

STOCKOUT TIME CALCULATION

In [80]:
# If historical data is not available, use a user-defined forecast growth rate from external research or expert opinions.
# industry research suggests a 3% monthly growth rate (0.03)
forecast_growth = 0.03
def calculate_stockout_time(row, forecast_growth=0):
    
    # Calculate adjusted demand rate
    adjusted_demand = row["Units Sold"] * (1 + forecast_growth)
    # Avoid division by zero
    if adjusted_demand == 0:
        adjusted_demand+=1
    return row["Inventory Level"] / adjusted_demand

# Apply the calculation across the DataFrame
df["Stockout Time (days)"] = df.apply(lambda row: calculate_stockout_time(row, forecast_growth), axis=1)


In [81]:
df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,2022,1,1,0.5,0.866025,1.765920
1,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,2022,1,1,0.5,0.866025,1.320388
2,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,2022,1,1,0.5,0.866025,1.523525
3,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,2022,1,1,0.5,0.866025,7.464587
4,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,2022,1,1,0.5,0.866025,11.511789


In [82]:
data = df.copy()

LABEL ENCODING

In [83]:
# List of categorical columns for label encoding:
categorical_columns = ["Store ID", "Product ID", "Category", "Region", 
                       "Weather Condition", "Seasonality"]

# Initialize a LabelEncoder
le = LabelEncoder()

# Apply label encoding to each categorical column:
for col in categorical_columns:
    df[col] = le.fit_transform(df[col])

df.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,0,0,3,1,231,127,55,135.47,33.50,20,1,0,29.69,0,2022,1,1,0.5,0.866025,1.765920
1,0,1,4,2,204,150,66,144.04,63.01,20,3,0,66.16,0,2022,1,1,0.5,0.866025,1.320388
2,0,2,4,3,102,65,51,74.02,27.99,10,3,1,31.32,2,2022,1,1,0.5,0.866025,1.523525
3,0,3,4,1,469,61,164,62.18,32.72,10,0,1,34.74,0,2022,1,1,0.5,0.866025,7.464587
4,0,4,1,0,166,14,135,9.26,73.64,0,3,0,68.95,2,2022,1,1,0.5,0.866025,11.511789


NORMALIZATION

In [84]:
numeric_columns = ["Inventory Level", "Units Sold", "Units Ordered", "Demand Forecast", 
                   "Price", "Discount", "Competitor Pricing"]

# Initialize the StandardScaler for normalization (z-score normalization)
scaler = MinMaxScaler(feature_range=(0,1))

# Fit and transform the numeric columns
df[numeric_columns] = scaler.fit_transform(df[numeric_columns])
df.head()




,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,M1,M2,Stockout Time (days)
0,0,0,3,1,0.402222,0.254509,0.194444,0.275211,0.261111,1.0,1,0,0.246822,0,2022,1,1,0.5,0.866025,1.765920
1,0,1,4,2,0.342222,0.300601,0.255556,0.291425,0.589000,1.0,3,0,0.611851,0,2022,1,1,0.5,0.866025,1.320388
2,0,2,4,3,0.115556,0.130261,0.172222,0.158947,0.199889,0.5,3,1,0.263137,2,2022,1,1,0.5,0.866025,1.523525
3,0,3,4,1,0.931111,0.122244,0.800000,0.136546,0.252444,0.5,0,1,0.297368,0,2022,1,1,0.5,0.866025,7.464587
4,0,4,1,0,0.257778,0.028056,0.638889,0.036421,0.707111,0.0,3,0,0.639776,2,2022,1,1,0.5,0.866025,11.511789


*MODEL TRAINING*

SPLITING TRAINING AND VALIDATION

In [85]:
features = df.drop(['Stockout Time (days)'], axis=1)
target = df['Stockout Time (days)'].values


X_train, X_val, Y_train, Y_val = train_test_split(features, target,
                                                  test_size = 0.05,
                                                  random_state=22)
X_train.shape, X_val.shape



((69445, 19), (3655, 19))

MODEL USED XGB

In [86]:
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Define XGBoost model
xgb_model = xgb.XGBRegressor()

# # Define hyperparameters to tune
# param_grid = {
#     'n_estimators': [100, 300, 500],  # Number of boosting rounds
#     'max_depth': [3, 5, 7],  # Depth of trees
#     'learning_rate': [0.01, 0.1, 0.3],  # Step size shrinkage
#     'subsample': [0.8, 1.0],  # Fraction of samples used per tree
#     'colsample_bytree': [0.8, 1.0],  # Fraction of features used per tree
# }

# # Use GridSearchCV
# grid_search = GridSearchCV(
#     estimator=xgb_model,
#     param_grid=param_grid,
#     scoring='neg_mean_squared_error',  # Minimize MSE
#     cv=3,  # 3-fold cross-validation
#     verbose=1,  # Print progress
#     n_jobs=-1  # Use all CPU cores
# )

# # Train GridSearchCV
# grid_search.fit(X_train, Y_train)

# # Best parameters
# print("Best Parameters:", grid_search.best_params_)

# # Train final model with best parameters
# best_model = xgb.XGBRegressor(**grid_search.best_params_)
# best_model.fit(X_train, Y_train)

# # Make predictions
# y_pred = best_model.predict(X_val)

# # Evaluate performance
# mse = mean_squared_error(Y_val, y_pred)
# print("Final Model MSE:", mse)


In [87]:
# # Train final model with best parameters
# best_model = xgb.XGBRegressor(**grid_search.best_params_)
# best_model.fit(X_train, Y_train)

# # Make predictions on test data
# y_pred = best_model.predict(X_val)

# # Evaluate model performance
# mse = mean_squared_error(Y_val, y_pred)
# print("Final Model MSE:", mse)


In [88]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Scatter plot of actual vs predicted values
# plt.figure(figsize=(8, 6))
# sns.scatterplot(x=Y_val, y=y_pred, alpha=0.7, edgecolor="k")
# plt.plot([min(Y_val), max(Y_val)], [min(Y_val), max(Y_val)], linestyle="--", color="red")  # Reference line
# plt.xlabel("Actual Values")
# plt.ylabel("Predicted Values")
# plt.title("Predicted vs. Actual Stockout Time")
# plt.show()


In [95]:
# best_model.save_model("xgb_stockout_model.json")

loaded_model = xgb.Booster()
loaded_model.load_model("xgb_stockout_model.json")
print(loaded_model.num_features())



19


In [90]:
import pandas as pd
import numpy as np

# Set a seed for reproducibility
np.random.seed(42)

# Number of sample rows to generate
num_rows = 10

# Create a DataFrame with similar columns and random data
df_sim = pd.DataFrame({
    "Store ID": np.random.randint(0, 5, size=num_rows),                # e.g., store IDs as integers
    "Product ID": np.random.randint(0, 100, size=num_rows),              # product IDs as integers
    "Category": np.random.randint(0, 5, size=num_rows),                  # category codes (e.g., 0 to 4)
    "Region": np.random.randint(0, 4, size=num_rows),                    # region codes (e.g., 0 to 3)
    "Inventory Level": np.round(np.random.uniform(0.35, 0.45, size=num_rows), 6),
    "Units Sold": np.round(np.random.uniform(0.24, 0.27, size=num_rows), 6),
    "Units Ordered": np.round(np.random.uniform(0.18, 0.20, size=num_rows), 6),
    "Demand Forecast": np.round(np.random.uniform(0.26, 0.28, size=num_rows), 6),
    "Price": np.round(np.random.uniform(0.25, 0.27, size=num_rows), 6),
    "Discount": np.round(np.random.uniform(0.95, 1.0, size=num_rows), 6),  # discount near 1.0
    "Weather Condition": np.random.randint(0, 4, size=num_rows),         # weather codes (e.g., 0 to 3)
    "Holiday/Promotion": np.random.randint(0, 2, size=num_rows),           # binary flag: 0 or 1
    "Competitor Pricing": np.round(np.random.uniform(0.24, 0.26, size=num_rows), 6),
    "Seasonality": np.random.randint(0, 3, size=num_rows),               # seasonality codes (e.g., 0 to 2)
    "Year": np.full(num_rows, 2025),                                     # constant year
    "Month": np.random.randint(1, 13, size=num_rows),                    # month 1 to 12
    "Day": np.random.randint(1, 29, size=num_rows),                      # day 1 to 28 (to avoid month-end issues)
    "M1": np.full(num_rows, 0.5),                                          # cyclical feature (sine value)
    "M2": np.full(num_rows, 0.866025)                                      # cyclical feature (cosine value)
})

preds = loaded_model.predict(xgb.DMatrix(df_sim))
print(preds)


[1.8311712 1.7657721 1.8699706 1.7694327 1.6322923 1.8981144 1.6298435
 1.7149049 1.8271909 2.0240667]
